# System Dependencies and Libraries

In [1]:
# Install System-level OCR
!apt-get update -qq
!apt-get install -y tesseract-ocr -qq

# Install Python ML and Web stack
!pip install -q qdrant-client sentence-transformers pymupdf pytesseract streamlit google-generativeai pillow

# Install Localtunnel to expose Streamlit to the web
!npm install -q localtunnel

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

# The RAG Backend

In [2]:
%%writefile backend.py
import os
import io
import time
import logging
from typing import List, Dict, Any
from concurrent.futures import ProcessPoolExecutor
import fitz
import pytesseract
from PIL import Image
import torch
from qdrant_client import QdrantClient
from qdrant_client.http import models
from sentence_transformers import SentenceTransformer, CrossEncoder

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- MULTIPROCESSING WORKER ---
# This MUST remain completely outside the class so it can be pickled across CPU cores
def _parallel_ocr_worker(pdf_path: str, page_num: int) -> tuple:
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_num)
    text = page.get_text("text").strip()

    # OCR Fallback for scanned pages
    if len(text) < 100:
        # CRITICAL OPTIMIZATION: Force matrix scale to 1.0 to shrink image byte size.
        # This makes Tesseract run 4x-5x faster while maintaining enough fidelity to read text.
        pix = page.get_pixmap(matrix=fitz.Matrix(1.0, 1.0))
        img_data = pix.tobytes("png")
        image = Image.open(io.BytesIO(img_data))
        text = pytesseract.image_to_string(image)

    doc.close()
    return (page_num + 1, text)

# --- ENTERPRISE PIPELINE ---
class ProductionRAGPipeline:
    def __init__(self, collection_name: str = "private_corpus"):
        self.collection_name = collection_name

        # Hardware Acceleration: Force models to use CUDA/T4 GPU if available
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        logger.info(f"Initializing AI Models on hardware: {self.device.upper()}")

        self.embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=self.device)
        self.reranker = CrossEncoder("BAAI/bge-reranker-base", device=self.device)

        self.db_client = QdrantClient(path="./qdrant_storage")
        self._setup_vector_collection()

    def _setup_vector_collection(self):
        if not self.db_client.collection_exists(self.collection_name):
            self.db_client.create_collection(
                collection_name=self.collection_name,
                vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE),
                hnsw_config=models.HnswConfigDiff(m=16, ef_construct=64, full_scan_threshold=10000, on_disk=False)
            )

    def chunk_text(self, text: str, chunk_size: int = 700, overlap: int = 150) -> List[str]:
        words = text.split()
        chunks = []
        i = 0
        while i < len(words):
            chunk = " ".join(words[i:i + chunk_size])
            chunks.append(chunk)
            i += (chunk_size - overlap)
        return chunks

    def ingest_pdf(self, pdf_path: str):
        doc = fitz.open(pdf_path)
        total_pages = len(doc)
        doc.close()

        logger.info(f"Starting Multi-Core Parallel Extraction for {total_pages} pages...")
        start_time = time.time()
        extracted_pages = []

        with ProcessPoolExecutor(max_workers=2) as executor:
            futures = [executor.submit(_parallel_ocr_worker, pdf_path, p) for p in range(total_pages)]
            for future in futures:
                extracted_pages.append(future.result())

        logger.info(f"Extraction complete in {round(time.time() - start_time, 2)}s. Generating GPU embeddings...")

        points = []
        point_id = int(time.time() * 1000)

        # --- STRUCTURAL TRACKING ADDED HERE ---
        current_chapter = "Front Matter / Introduction"

        for page_num, text in extracted_pages:
            if not text: continue

            # Simple heuristic: If the word "Chapter" appears early on the page, grab that line
            if "chapter" in text[:200].lower():
                first_line = text[:200].strip().split('\n')
                if len(first_line) < 50: # Ensure it's actually a title and not just a sentence
                    current_chapter = first_line

            chunks = self.chunk_text(text)
            for chunk_content in chunks:
                normalized_text = " ".join(chunk_content.replace("\n", " ").split())
                vector = self.embedding_model.encode(normalized_text).tolist()

                # We inject the 'current_chapter' into the metadata payload
                payload = {
                    "filename": os.path.basename(pdf_path),
                    "page_number": page_num,
                    "chapter": current_chapter, # Added Chapter Metadata
                    "text": f"[Section: {current_chapter}] " + normalized_text
                }

                points.append(models.PointStruct(id=point_id, vector=vector, payload=payload))
                point_id += 1

        if points:
            self.db_client.upload_points(collection_name=self.collection_name, points=points)
        logger.info(f"✅ Ingestion successful. Vectors indexed with structural metadata.")

    def query_pipeline(self, query_text: str, top_k_ann: int = 15, top_k_rerank: int = 4) -> Dict[str, Any]:
        start_time = time.time()

        query_vector = self.embedding_model.encode(query_text).tolist()

        # FIXED: Using the modern Qdrant API method (query_points instead of search)
        search_response = self.db_client.query_points(
            collection_name=self.collection_name,
            query=query_vector,
            limit=top_k_ann
        )
        ann_results = search_response.points  # Extract the matched points

        if not ann_results:
            return {"context_window": "", "sources": [], "latency_seconds": time.time() - start_time}

        passages = [hit.payload["text"] for hit in ann_results]
        pairs = [[query_text, passage] for passage in passages]
        rerank_scores = self.reranker.predict(pairs)

        for idx, score in enumerate(rerank_scores):
            ann_results[idx].score = float(score)

        reranked_results = sorted(ann_results, key=lambda x: x.score, reverse=True)[:top_k_rerank]

        context_str = "\n\n".join([f"--- Context {i+1} ---\n{hit.payload['text']}" for i, hit in enumerate(reranked_results)])
        sources = [{"filename": hit.payload["filename"], "page": hit.payload["page_number"]} for hit in reranked_results]

        return {
            "context_window": context_str,
            "sources": sources,
            "latency_seconds": round(time.time() - start_time, 3)
        }

Writing backend.py


# The UI & Frontend App

In [3]:
%%writefile app.py
import streamlit as st
import google.generativeai as genai
import time
import os
from backend import ProductionRAGPipeline

st.set_page_config(page_title="Enterprise RAG System", layout="centered")

@st.cache_resource
def load_backend():
    return ProductionRAGPipeline()

rag_backend = load_backend()

# --- Sidebar Configuration ---
with st.sidebar:
    st.header("⚙️ Configuration")
    api_key = st.text_input("Gemini API Key", type="password", help="Get this from Google AI Studio")

    st.divider()
    st.header("📄 Document Ingestion")
    uploaded_files = st.file_uploader("Upload PDFs (≥200 pages)", type="pdf", accept_multiple_files=True)

    if st.button("Run Ingestion Pipeline"):
        if not uploaded_files:
            st.error("Please upload files first.")
        else:
            with st.spinner("Extracting, Chunking, and Embedding..."):
                for file in uploaded_files:
                    file_path = f"./{file.name}"
                    with open(file_path, "wb") as f:
                        f.write(file.getbuffer())
                    rag_backend.ingest_pdf(file_path)
                st.success("✅ Vectors Synced to Qdrant!")

# --- Main Chat UI ---
st.title("📚 Private Knowledge RAG")
st.caption("Hybrid Search (HNSW ANN + Cross-Encoder) with Google Gemini")

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if user_query := st.chat_input("Query the database..."):
    if not api_key:
        st.error("⚠️ API Key required. Please configure it in the sidebar.")
        st.stop()

    genai.configure(api_key=api_key)
    generation_model = genai.GenerativeModel('gemini-2.5-flash')

    with st.chat_message("user"):
        st.markdown(user_query)
    st.session_state.messages.append({"role": "user", "content": user_query})

    with st.chat_message("assistant"):
        with st.spinner("Executing Vector Search & Semantic Reranking..."):
            retrieval_data = rag_backend.query_pipeline(user_query)
            context = retrieval_data["context_window"]
            sources = retrieval_data["sources"]
            search_latency = retrieval_data["latency_seconds"]

            if not sources:
                response_text = "No relevant context found in the database."
            else:
                prompt = f"""
                You are a highly accurate analytical assistant. Use ONLY the provided context to answer the user's question.
                If the answer is not contained in the context, say "I don't have enough information."
                Question: {user_query}
                Context Documents: {context}
                """

                start_gen = time.time()
                llm_response = generation_model.generate_content(prompt)
                gen_latency = time.time() - start_gen
                response_text = llm_response.text

            total_time = round(search_latency + gen_latency, 2)
            st.markdown(response_text)

            if sources:
                st.divider()
                st.caption(f"⏱️ **Pipeline Latency:** {total_time}s (Retrieval: {search_latency}s | LLM: {round(gen_latency, 2)}s)")
                st.caption("**Verified Citations:**")
                # Deduplicate sources for clean UI
                unique_sources = {f"{src['filename']} (Page {src['page']})" for src in sources}
                for src in unique_sources:
                    st.caption(f"- 📄 `{src}`")

    st.session_state.messages.append({"role": "assistant", "content": response_text})

Writing app.py


# Security Endpoint Bypass

In [4]:
import urllib.request
print("COPY THIS IP FOR THE NEXT STEP:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

COPY THIS IP FOR THE NEXT STEP: 34.7.87.228


# Launch the Application

In [6]:
# 1. Native Linux command to forcibly kill ANYTHING on port 8501 without asking
!fuser -k -9 8501/tcp
!pkill -9 -f streamlit
!pkill -9 -f cloudflared

# 2. Clear old logs
!rm -f streamlit_run.log cloudflare_tunnel.log

# 3. Start Streamlit exactly on port 8501
!nohup streamlit run app.py --server.port 8501 --server.enableCORS=false --server.enableXsrfProtection=false > streamlit_run.log 2>&1 &
print("--- Streamlit Force-Started ---")

# 4. Wait 4 seconds for Streamlit to fully bind to the port
!sleep 4

# 5. Start Cloudflare Tunnel
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8501 > cloudflare_tunnel.log 2>&1 &

# 6. Wait 6 seconds, then print the link
!sleep 6
print("\n--- ✅ CLICK YOUR FINAL CLOUDFLARE LINK BELOW ✅ ---")
!grep -o 'https://.*trycloudflare.com' cloudflare_tunnel.log || echo "⚠️ Link not ready yet, wait 3 seconds and run !grep -o 'https://.*trycloudflare.com' cloudflare_tunnel.log"

8501/tcp:             2583
--- Streamlit Force-Started ---

--- ✅ CLICK YOUR FINAL CLOUDFLARE LINK BELOW ✅ ---
⚠️ Link not ready yet, wait 3 seconds and run !grep -o 'https://.*trycloudflare.com' cloudflare_tunnel.log
